# **PHYSIOLOGICAL SIGNALS PREPROCESSING**
For physiological signals captured by Shimmer3 GSR+ and EmotiBit devices.

|DEVICE |SIGNAL | UNIT OF MEASUREMENT | SAMPLING FREQUENCY |
|----------|----------|----------|----------|
| SHIMMER3 GSR+	| PPG | [milliVolts](https://shimmersensing.com/wp-content/docs/support/documentation/Optical_Pulse_Sensor_User_Guide_rev1.6.pdf)	| 100.21 Hz |
| |	EDA/GSR	| micro Siemens (µS) | 100.21 Hz|
| EMOTIBIT |	PPG (3 channels) | [amount of light](https://www.reddit.com/r/EmotiBit/comments/112ao2g/ppg_raw_units/?rdt=57838) |	25 Hz |
| | EDA/GSR|	microSiemens (µS) |	15 Hz |


## Functions and variables definitions

In [1]:
import inspect
import pickle
import numpy as np
import pandas as pd
import pandas as pd
import neurokit2 as nk
import numpy as np
from tqdm import tqdm

In [2]:
EDA_FREQ = 15 # Final frequency after resampling
BVP_FREQ = 50

In [3]:

def extract_native_freq(signal, device):
    native_sampling_rates = {
        "emotibit": {"Gsr": 15, "Bvp": 25},
        "shimmer": {"Gsr": 100.21, "Bvp": 100.21},
    }

    device_lower = device.lower()
    if "emotibit" in device_lower and signal in native_sampling_rates["emotibit"]:
        return native_sampling_rates["emotibit"][signal]
    elif "shimmer" in device_lower and signal in native_sampling_rates["shimmer"]:
        return native_sampling_rates["shimmer"][signal]
    else:
        return None


def extract_new_freq(signal, device):
    new_sampling_rates = {
        "emotibit": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ},
        "shimmer": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ},
    }

    device_lower = device.lower()
    if "emotibit" in device_lower and signal in new_sampling_rates["emotibit"]:
        return new_sampling_rates["emotibit"][signal]
    elif "shimmer" in device_lower and signal in new_sampling_rates["shimmer"]:
        return new_sampling_rates["shimmer"][signal]
    else:
        return None

In [4]:
from scipy.interpolate import InterpolatedUnivariateSpline

def resample_with_spline(df, target_frequency=15, timestamp_col="Timestamp", signal_col="Gsr"):
    # Ensure order and remove duplicates
    df = df.drop_duplicates(subset=[timestamp_col])

    # Sampling interval in seconds
    target_interval = 1 / target_frequency

    # Timestamps and signal
    timestamps = df[timestamp_col].values
    signal = df[signal_col].values

    # Uniform timestamps for resampling
    new_timestamps = np.arange(
        timestamps.min(), timestamps.max(), target_interval)

    # Cubic spline interpolator
    spline = InterpolatedUnivariateSpline(timestamps, signal, k=3)
    new_signal = spline(new_timestamps)

    # Create resampled DataFrame with the same column names
    resampled_df = pd.DataFrame({
        timestamp_col: new_timestamps,
        signal_col: new_signal
    })

    return resampled_df

## Pipeline


The dictionary structure once the pipeline is finished should be something like this:

```json
{
    "shimmer": {
        "user1": {
            "Gsr": {
                "original": {
                    "df": "DataFrame with original data",
                },
                "hampel+IQR": {
                    "original": {
                        "df": "DataFrame without outliers",
                    },
                    "butterworth": {
                        "original": {
                            "df": "DataFrame filtered with Butterworth",
                        }
                    },
                    "gaussian": {
                        "original": {
                            "df": "DataFrame filtered with Gaussian",
                        }
                    }
                }
            },
            "Bvp": {
                "original": {
                    "df": "DataFrame with original data",
                },
                "hampel+IQR": {
                    "original": {
                        "df": "DataFrame without outliers",
                    },
                    "butterworth": {
                        "original": {
                            "df": "DataFrame filtered with Butterworth",
                        }
                    },
                    "fourth_cheby2": {
                        "original": {
                            "df": "DataFrame filtered with Chebyshev II",
                        }
                    }
                }
            }
        }
    }
}




### Defining steps

In [5]:
class FlexibleDict(dict):
    """
    A dictionary subclass that adds flexible key handling.

    Features:
    1. Allows tuples as keys and automatically expands them into multiple individual keys 
       with the same value.
    2. Supports partial key matching: if an exact key is not found, it checks if any 
       existing key is contained within the requested key.
    3. Falls back to a special key "other" if no match is found.
    """

    def __init__(self, *args, **kwargs):
        """
        Initialize the FlexibleDict.

        Args:
            *args: Positional arguments for the base dict initialization.
            **kwargs: Keyword arguments for the base dict initialization.

        Behavior:
            - If any key is a tuple, it creates individual keys for each element in the tuple
              with the same value.
            - Deletes the original tuple key afterwards.
        """
        super().__init__(*args, **kwargs)

        for key in list(self.keys()):
            if isinstance(key, tuple):
                for sub_key in key:
                    # Assign the same value to each sub_key
                    self[sub_key] = self[key]
                del self[key]  # Remove the original tuple key

    def __getitem__(self, key):
        """
        Retrieve an item from the dictionary with flexible key handling.

        Args:
            key: The key to retrieve.

        Returns:
            The value associated with the key.

        Behavior:
            - If the exact key exists, returns its value.
            - If no exact match, checks if any existing key is a substring of the requested key.
            - If no partial match is found, returns the value for the key "other" if it exists.
            - Otherwise, raises a KeyError.
        """
        if key in self:
            return super().__getitem__(key)

        # Check partial matches
        for k in self.keys():
            if k in key:
                return self[k]

        # Fallback to "other"
        if "other" in self:
            return self["other"]

        raise KeyError(f"Key {key} not found")


def manage_parameters(func, df, column, freq):
    """
    Call a function with a flexible number of parameters.

    Args:
        func (callable): The function to call.
        df (pd.DataFrame): A DataFrame to pass to the function.
        column (str): A column name in the DataFrame.
        freq (float): Frequency.

    Returns:
        The result of calling `func` with the appropriate parameters.

    Behavior:
        - If the function expects 2 parameters, calls `func(df, column)`.
        - If the function expects 3 parameters, calls `func(df, column, freq)`.
        - Otherwise, raises a ValueError indicating an unexpected number of parameters.
    """
    params = inspect.signature(func).parameters
    if len(params) == 2:
        return func(df, column)
    elif len(params) == 3:
        return func(df, column, freq)
    else:
        raise ValueError(
            f'Function {func.__name__} has an unexpected number of parameters.')

In [6]:

from utils.outliers import hampel_IQR_GSR_BVP, IQR

step1 = {
    "name": "outliers",
    "functions": FlexibleDict({
        "gsr": {"hampel+IQR": hampel_IQR_GSR_BVP,
                },
        ("bvp", "pi", "pr"): {
                "hampel+IQR": hampel_IQR_GSR_BVP,
                },
        "other": {
            "IQR": IQR,
            }
        }
    )
}

from utils.filtering import four_cheby2_bvp, butterworth_bvp, langevin_bandpass, butterworth_gsr, gaussian_gsr, five_cheby2_gsr, emotibit_filter

step2 = {
    "name": "filtering",
    "functions": FlexibleDict({
        "gsr": {
            "butterworth": butterworth_gsr,
            "gaussian": gaussian_gsr,
            "5cheby2": five_cheby2_gsr
            },
        ("bvp", "pi", "pr"): {
            "emotibit": emotibit_filter,
            "4cheby2": four_cheby2_bvp,
            "butterworth": butterworth_bvp,
            "langevin": langevin_bandpass,
            },
        "other": {
        }
    }
    )
}

steps = [step1, step2]

In [7]:
def apply_steps(ref, column, device, steps, current_step=0, extract_freq=extract_new_freq):
    """
    Recursively apply processing steps.

    Args:
        ref (dict): Nested dict containing the DataFrame under `ref["original"]["df"]`.
        column (str): Column name to process.
        device (str): Device name identifier.
        steps (list): List of steps.
        current_step (int, optional): Current step index. Defaults to 0.
        extract_freq (callable, optional): Function to get sampling rate.

    """

    if current_step == len(steps):
        return

    df = ref["original"]["df"].copy()
    functions_to_apply = steps[current_step]["functions"][column.lower(
    )] | steps[current_step]["functions"]["other"]
    sampling_rate = extract_freq(column, device)

    for func_name, func in functions_to_apply.items():
        ref[func_name] = {"original": {}}
        ref[func_name]["original"]["df"] = manage_parameters(
            func, df.copy(), column, sampling_rate)
        apply_steps(ref[func_name], column, device, steps,
                    current_step + 1, extract_freq=extract_freq)

# Heart Rate derivation

In [8]:
from utils.filtering import DigitalFilter

def hr_emotibit(df, freq):
    timestamps = df["Timestamp"].values
    heartRateFilter = DigitalFilter("IIR_LOWPASS", freq, 1)

    timePeriod = (1.0 / freq) * 1000

    peaks = nk.ppg.ppg_findpeaks(
        df[df.columns[-1]].values, sampling_rate=freq, method="elgendi")["PPG_Peaks"]

    beats = np.zeros(len(df[df.columns[-1]].values), dtype=int)
    beats[peaks] = 1

    heartRate = []
    beat_timestamps = []

    interBeatSampleCount = 0
    for i in range(len(beats)):
        interBeatSampleCount += 1

        if beats[i]:
            interBeatInterval = interBeatSampleCount * timePeriod

            heart_rate = (60.0 / interBeatInterval) * 1000
            heart_rate = heartRateFilter.filter(heart_rate)
            heartRate.append(heart_rate)

            beat_timestamps.append(timestamps[i])

            interBeatSampleCount = 0

    return pd.DataFrame({"Timestamp": beat_timestamps, "Hr_emotibit_e": heartRate})

In [9]:
import copy

def calculate_hr(data_pipeline, hr_func, hr_field, extract_freq=extract_new_freq):
    """
    Calculate heart rate (HR) from a base PPG/BVP signal.

    Args:
        data_pipeline (dict): Nested pipeline structure containing DataFrames per device/user.
        hr_func (callable): HR processing function (e.g., hr_neurokit).
        hr_field (str): Name of the HR field to create (e.g., "Hr").
        extract_freq (callable, optional): Function to get signal sampling frequency.

    """

    def process_hr_branch(ref):
        """Recursively process each branch of the pipeline."""
        if "df" in ref:
            return

        for technique in ref:
            if technique == "original":
                timestamps = ref[technique]["df"]["Timestamp"].values
                try:
                    new_df = hr_func(ref[technique]["df"], freq)
                    ref[technique]["df"] = new_df
                except Exception as e:
                    print("No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...")
                    ref[technique]["df"] = pd.DataFrame({
                        "Timestamp": timestamps,
                        hr_field: [40] * len(timestamps)  # fallback HR
                    })
            else:
                process_hr_branch(ref[technique])

    for device in data_pipeline.keys():
        freq = extract_freq("Bvp", device)
        print(f"Calculating HR for device: {device} with freq: {freq} Hz")

        for user in data_pipeline[device]:
            try:
                # Initialize HR branch by copying the original BVP signal
                data_pipeline[device][user][hr_field] = copy.deepcopy(
                    data_pipeline[device][user]["Bvp"]
                )
            except KeyError:
                continue

            process_hr_branch(data_pipeline[device][user][hr_field])

## Execution

In [10]:
from utils.loader import load_data

devices_global, users_list, data, common_columns = load_data(
    f"./data")
print("Loaded dataset.")
print("Devices:", devices_global)
print("Users:", users_list)

# --- Resample signals ---
print("\nResampling signals...")
for device in devices_global:
    print(f" Device: {device}")
    for user in users_list:
        print(f"   User: {user}")

        signals_to_process = ["Gsr", "Bvp"]

        for signal in signals_to_process:
            df = data[device][user].get(signal)
            if df is None:
                print(f"     Signal {signal} missing, skipping.")
                continue

            native_freq = extract_native_freq(signal, device)
            target_freq = EDA_FREQ if signal == "Gsr" else BVP_FREQ

            print(
                f"     Signal {signal}: native {native_freq} Hz, target {target_freq} Hz")

            if target_freq != native_freq:
                print(f"     Resampling {signal}...")
                df = resample_with_spline(
                    df,
                    target_frequency=target_freq,
                    timestamp_col="Timestamp",
                    signal_col=signal
                )
            else:
                print(f"     No resampling needed for {signal}")

            data[device][user][signal] = df

# --- Build data pipeline ---
print("\nBuilding data pipeline...")
data_pipeline = {}
for device in devices_global:
    print(f" Device: {device}")
    if device not in data_pipeline:
        data_pipeline[device] = {}
    for user in data[device]:
        print(f"   User: {user}")
        data_pipeline[device][user] = {}
        for column in data[device][user]:
            print(f"     Column: {column}")

            sampling_rate = extract_new_freq(column, device)

            data_pipeline[device][user][column] = {
                "original": {
                    "df": data[device][user][column].copy()
                }
            }

# --- Apply steps ---
print("\nApplying steps...")
for device in tqdm(devices_global):
    print(f"\nDevice: {device}")
    for user in tqdm(data[device]):
        print(f"  User: {user}")
        for column in data[device][user]:
            print(f"    Processing column: {column}")

            apply_steps(
                data_pipeline[device][user][column],
                column,
                device,
                steps,
                0,
                extract_freq=extract_new_freq
            )

# --- Calculate heart rate ---
print("\nCalculating heart rate...")
calculate_hr(
    data_pipeline,
    hr_emotibit,
    "Hr",
    extract_freq=extract_new_freq
)
print("Heart rate calculation done.")

# --- Save processed data ---
print("\nSaving processed data...")
with open(f"processed_data.pickle", "wb") as file:
    pickle.dump(data_pipeline, file)
print(f"Saved to processed_data.pickle")

print("\nFinished dataset.\n")

Loaded dataset.
Devices: ['emotibit_dorsal', 'emotibit_volar', 'shimmer_fingers', 'shimmer_wrist']
Users: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

Resampling signals...
 Device: emotibit_dorsal
   User: 1
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 50 Hz
     Resampling Bvp...
   User: 2
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 50 Hz
     Resampling Bvp...
   User: 3
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 50 Hz
     Resampling Bvp...
   User: 4
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 50 Hz
     Resampling Bvp...
   User: 5
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 50 Hz
     Resampling Bvp...
   User: 6
   

  0%|          | 0/4 [00:00<?, ?it/s]


Device: emotibit_dorsal


  User: 1
    Processing column: Bvp
    Processing column: Gsr


  User: 2
    Processing column: Bvp
    Processing column: Gsr


...

  User: 7
    Processing column: Bvp
    Processing column: Gsr


  User: 8
    Processing column: Bvp
    Processing column: Gsr


  User: 9
    Processing column: Bvp
    Processing column: Gsr


100%|██████████| 4/4 [02:24<00:00, 36.17s/it]



Calculating heart rate...
Calculating HR for device: emotibit_dorsal with freq: 50 Hz
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate could be calculated: no peaks detected. The PPG/BVP signal may not have been filtered at this step. Continuing...
No heart rate cou